# Zeta RL — Kaggle Training Notebook

Run SAC or PPO training on a Kaggle T4/P100 GPU (30 GPU hrs/week free).  
Sessions cut off at 12 hours; use **Cell 5** to resume from the last checkpoint.

**Before running:**
1. Enable GPU: *Settings → Accelerator → GPU T4 x2 (or P100)*
2. Add your wandb key: *Settings → Secrets → Add → Name: `WANDB_API_KEY`, Value: your key*
3. Run all cells top-to-bottom for a fresh run, or skip Cell 4 and run Cell 5 to resume.

In [ ]:
# Cell 1 — Install dependencies
# torch is already present on Kaggle GPU images; install the rest.
!pip install -q \
    gymnasium>=1.0 \
    hydra-core>=1.3 \
    omegaconf>=2.3 \
    wandb>=0.17 \
    stable-baselines3>=2.3 \
    imageio>=2.34 \
    'imageio-ffmpeg>=0.5'

In [ ]:
# Cell 2 — Clone repo and install as editable package
import os

REPO_URL = "https://github.com/badkoubeh/zeta-rl.git"
REPO_DIR = "/kaggle/working/zeta-rl"
RESULTS_DIR = "/kaggle/working/results"

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    print("Repo already cloned; pulling latest.")
    !git -C {REPO_DIR} pull --ff-only

!pip install -q -e "{REPO_DIR}[train]"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Make the repo the working directory so Hydra finds configs/
os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

In [ ]:
# Cell 3 — Configure wandb
# The key is pulled from Kaggle Secrets (no hardcoded credentials).
from kaggle_secrets import UserSecretsClient
import os

try:
    secret = UserSecretsClient().get_secret("WANDB_API_KEY")
    os.environ["WANDB_API_KEY"] = secret
    os.environ["WANDB_MODE"] = "online"
    print("wandb: online mode, key loaded from Kaggle Secrets")
except Exception:
    os.environ["WANDB_MODE"] = "offline"
    print("wandb: offline mode (WANDB_API_KEY secret not found)")

In [ ]:
# Cell 4 — Fresh training run
# Adjust agent (sac|ppo), seed, and total_steps as needed.
# Checkpoints save every 50k steps to /kaggle/working/results/ (persists as Kaggle output).
# If the session is cut off before training finishes, use Cell 5 to resume.

AGENT = "sac"      # sac | ppo
SEED  = 42
STEPS = 2_000_000

!python experiments/train.py \
    agent={AGENT} \
    seed={SEED} \
    total_steps={STEPS} \
    compute=small_gpu \
    results_dir={RESULTS_DIR}/{AGENT}_moderate_nominal_{SEED}

In [ ]:
# Cell 5 — Resume from latest checkpoint
# Run this cell (instead of Cell 4) at the start of a new session to continue
# a run that was interrupted. Point CHECKPOINT_PATH to the last .zip saved in
# /kaggle/working/results/ (add the previous session's output as a Kaggle dataset
# and mount it, or keep results inside /kaggle/working/ across sessions).

import glob
import os

AGENT = "sac"      # must match the original run
SEED  = 42
STEPS = 2_000_000  # same total; training continues for the remaining steps

run_dir = f"{RESULTS_DIR}/{AGENT}_moderate_nominal_{SEED}"

# Pick the highest-step checkpoint automatically
checkpoints = sorted(glob.glob(f"{run_dir}/*.zip"))
if not checkpoints:
    raise FileNotFoundError(
        f"No checkpoints found in {run_dir}. "
        "Add the previous session's output as a dataset and mount it."
    )
latest = checkpoints[-1]
print(f"Resuming from: {latest}")

!python experiments/train.py \
    agent={AGENT} \
    seed={SEED} \
    total_steps={STEPS} \
    compute=small_gpu \
    resume_from={latest} \
    results_dir={run_dir}